# NOTEBOOK 4 — PATTERN ANALYSIS & RESEARCH FINDINGS
**Project:** Trust & Transparency in AI-Powered Recruitment Platforms

**Primary prediction:** `bert_fraud_prediction` (DistilBERT + hybrid guard)  
**Confidence signal:** `bert_fraud_probability` + `confidence_label`  
**Risk signal:** `fraud_risk_pct` + `risk_tier`  

**Input:** `../data/predictions/combined_jobs_final_predictions.csv`  
**Outputs:** `../outputs/plots/`, `../outputs/reports/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

ROOT         = Path('..').resolve()
INPUT_PATH   = ROOT / 'data' / 'predictions' / 'combined_jobs_final_predictions.csv'
PLOTS_PATH   = ROOT / 'outputs' / 'plots'
REPORTS_PATH = ROOT / 'outputs' / 'reports'

PLOTS_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
df['posted_date']    = pd.to_datetime(df['posted_date'],    errors='coerce', utc=True)
df['collected_date'] = pd.to_datetime(df['collected_date'], errors='coerce', utc=True)
print(f"Rows loaded: {len(df)}")

PRED_COL    = 'bert_fraud_prediction'   # final binary label (0/1)
PROB_COL    = 'bert_fraud_probability'  # confidence score
SCORE_COL   = 'fraud_risk_pct'          # rule-based score
RISK_COLORS = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}

total_fake = df[PRED_COL].sum()
print(f"\n{'='*45}")
print(f"Fake jobs detected    : {total_fake} ({total_fake/len(df)*100:.1f}%)")
print(f"Likely Legitimate     : {len(df)-total_fake} ({(len(df)-total_fake)/len(df)*100:.1f}%)")
print(f"{'='*45}")
print(f"\nConfidence breakdown:")
print(df['confidence_label'].value_counts().to_string())
print(f"\nRisk tier distribution:")
print(df['risk_tier'].value_counts().to_string())

In [ ]:
# ANALYSIS 1: PLATFORM COMPARISON
print("ANALYSIS 1: PLATFORM COMPARISON")

plat = df.groupby('source_platform').agg(
    total_jobs        = (PRED_COL, 'count'),
    fake_count        = (PRED_COL, 'sum'),
    avg_risk_score    = (SCORE_COL, 'mean'),
    avg_bert_prob     = (PROB_COL, 'mean'),
    pct_missing_co    = ('missing_company', 'mean'),
    pct_high_salary   = ('high_salary_flag', 'mean'),
    pct_vague_loc     = ('vague_location', 'mean'),
    pct_contract_miss = ('contract_missing', 'mean'),
    pct_suspicious_kw = ('suspicious_keywords', 'mean'),
).reset_index()

plat['fake_rate_%']    = (plat['fake_count'] / plat['total_jobs'] * 100).round(1)
plat['avg_risk_score'] = plat['avg_risk_score'].round(1)
plat['avg_bert_prob']  = (plat['avg_bert_prob'] * 100).round(1)

print(plat[['source_platform','total_jobs','fake_count','fake_rate_%',
            'avg_risk_score']].to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analysis 1: Platform Comparison\nTrust & Transparency in AI-Powered Recruitment',
             fontsize=13, fontweight='bold')

ax = axes[0, 0]
bars = ax.bar(plat['source_platform'], plat['fake_rate_%'],
              color=sns.color_palette('muted', len(plat)))
ax.set_title('Fake Job Rate by Platform (%)', fontweight='bold')
ax.set_ylabel('Fake Rate %')
for b, v in zip(bars, plat['fake_rate_%']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02,
            f'{v}%', ha='center', fontweight='bold')

ax = axes[0, 1]
bars = ax.bar(plat['source_platform'], plat['avg_risk_score'],
              color=sns.color_palette('Set2', len(plat)))
ax.set_title('Average Rule-Based Risk Score by Platform', fontweight='bold')
ax.set_ylabel('Risk Score (%)')
for b, v in zip(bars, plat['avg_risk_score']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, str(v), ha='center', fontweight='bold')

ax = axes[1, 0]
feature_cols   = ['pct_missing_co','pct_high_salary','pct_vague_loc','pct_contract_miss','pct_suspicious_kw']
feature_labels = ['Missing Company','High Salary','Vague Location','Contract Missing','Susp. Keywords']
x = np.arange(len(feature_labels))
width = 0.35
palette = sns.color_palette('muted', len(plat))
for i, (_, row) in enumerate(plat.iterrows()):
    vals   = [row[c]*100 for c in feature_cols]
    offset = (i - len(plat)/2 + 0.5) * width
    ax.bar(x + offset, vals, width, label=row['source_platform'], color=palette[i], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(feature_labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('% Flagged')
ax.set_title('Fraud Indicator Rates by Platform', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1, 1]
tier_by_plat = df.groupby(['source_platform','risk_tier']).size().unstack(fill_value=0)
for col in ['Low','Medium','High']:
    if col not in tier_by_plat.columns:
        tier_by_plat[col] = 0
tier_pct = tier_by_plat[['Low','Medium','High']].div(
    tier_by_plat[['Low','Medium','High']].sum(axis=1), axis=0) * 100
tier_pct.plot(kind='bar', ax=ax, stacked=True,
              color=['#4CAF50','#FF9800','#F44336'], edgecolor='white')
ax.set_title('Risk Tier Distribution by Platform', fontweight='bold')
ax.set_ylabel('%'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Risk Tier', fontsize=8)

plt.tight_layout()
plt.savefig(PLOTS_PATH / '4_platform_analysis.png', bbox_inches='tight')
plt.close()
print("Saved → 4_platform_analysis.png")

In [ ]:
# ANALYSIS 2: CATEGORY / SECTOR ANALYSIS
print("ANALYSIS 2: CATEGORY/SECTOR ANALYSIS")

cat = df.groupby('category').agg(
    total_jobs      = (PRED_COL, 'count'),
    fake_count      = (PRED_COL, 'sum'),
    avg_risk_score  = (SCORE_COL, 'mean'),
    avg_bert_prob   = (PROB_COL, 'mean'),
    avg_salary      = ('salary_avg', 'mean'),
    pct_missing_co  = ('missing_company', 'mean'),
    pct_high_salary = ('high_salary_flag', 'mean'),
).reset_index()

cat['fake_rate_%']    = (cat['fake_count'] / cat['total_jobs'] * 100).round(1)
cat['avg_risk_score'] = cat['avg_risk_score'].round(1)
cat = cat[cat['total_jobs'] >= 10].sort_values('avg_risk_score', ascending=False)

print(cat[['category','total_jobs','fake_count','fake_rate_%',
           'avg_risk_score']].to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Analysis 2: Job Category / Sector Analysis', fontsize=13, fontweight='bold')

ax = axes[0, 0]
top_cat  = cat.nlargest(10, 'avg_risk_score')
colors_c = ['#F44336' if v > 15 else '#FF9800' if v > 8 else '#4CAF50'
            for v in top_cat['avg_risk_score']]
ax.barh(top_cat['category'][::-1], top_cat['avg_risk_score'][::-1], color=colors_c[::-1])
ax.set_title('Top 10 Categories by Avg Risk Score', fontweight='bold')
ax.set_xlabel('Average Risk Score (%)')
ax.axvline(8, color='orange', linestyle='--', alpha=0.7, label='Moderate risk')
ax.legend(fontsize=8)

ax = axes[0, 1]
top_fake = cat.nlargest(10, 'fake_rate_%')
ax.barh(top_fake['category'][::-1], top_fake['fake_rate_%'][::-1], color='tomato', alpha=0.8)
ax.set_title('Top 10 Categories by Fake Job Rate (%)', fontweight='bold')
ax.set_xlabel('Fake Rate %')
for i, (_, row) in enumerate(top_fake[::-1].iterrows()):
    ax.text(row['fake_rate_%']+0.05, i, f"{row['fake_rate_%']:.1f}%", va='center', fontsize=8)

ax = axes[1, 0]
sc = ax.scatter(cat['total_jobs'], cat['fake_rate_%'],
                s=cat['avg_risk_score']*8, alpha=0.6,
                c=cat['avg_risk_score'], cmap='RdYlGn_r')
plt.colorbar(sc, ax=ax, label='Avg Risk Score')
for _, row in cat.nlargest(5, 'avg_risk_score').iterrows():
    ax.annotate(row['category'][:20],
                (row['total_jobs'], row['fake_rate_%']), fontsize=7, ha='left')
ax.set_xlabel('Total Jobs in Category')
ax.set_ylabel('Fake Job Rate %')
ax.set_title('Volume vs Fake Rate\n(bubble size = avg risk score)', fontweight='bold')

ax = axes[1, 1]
salary_cat = cat.nlargest(10, 'total_jobs').sort_values('avg_salary', ascending=False)
salary_cat = salary_cat[salary_cat['avg_salary'].notna()]
ax.barh(salary_cat['category'][::-1], salary_cat['avg_salary'][::-1],
        color='steelblue', alpha=0.8)
ax.set_title('Average Salary by Category\n(Top 10 by volume)', fontweight='bold')
ax.set_xlabel('Average Salary (GBP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))

plt.tight_layout()
plt.savefig(PLOTS_PATH / '4_category_analysis.png', bbox_inches='tight')
plt.close()
print("Saved → 4_category_analysis.png")

In [ ]:
# ANALYSIS 3: SALARY PATTERN ANALYSIS
print("ANALYSIS 3: SALARY PATTERN ANALYSIS")

sal_df = df[df['salary_avg'].notna()].copy()

sal_tier = sal_df.groupby('risk_tier')['salary_avg'].agg(['mean','median','std','count']).round(0)
print("\nSalary stats by risk tier:")
print(sal_tier.to_string())

sal_pred = sal_df.groupby(PRED_COL)['salary_avg'].agg(['mean','median','count']).round(0)
sal_pred.index = ['Legitimate','Fake']
print("\nSalary stats by prediction:")
print(sal_pred.to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analysis 3: Salary Pattern Analysis', fontsize=13, fontweight='bold')

ax = axes[0, 0]
for tier, color in RISK_COLORS.items():
    data = sal_df[sal_df['risk_tier'] == tier]['salary_avg'].clip(upper=200000)
    if len(data):
        ax.hist(data, bins=30, alpha=0.6, color=color,
                label=f'{tier} (n={len(data)})', edgecolor='white')
ax.set_title('Salary Distribution by Risk Tier', fontweight='bold')
ax.set_xlabel('Salary (GBP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.legend()

ax = axes[0, 1]
groups = [sal_df[sal_df[PRED_COL]==0]['salary_avg'].clip(upper=200000),
          sal_df[sal_df[PRED_COL]==1]['salary_avg'].clip(upper=200000)]
bp = ax.boxplot(groups, labels=['Legitimate','Fake'], patch_artist=True,
                boxprops=dict(alpha=0.7))
bp['boxes'][0].set_facecolor('steelblue')
if len(bp['boxes']) > 1:
    bp['boxes'][1].set_facecolor('tomato')
ax.set_title('Salary Range: Legitimate vs Fake', fontweight='bold')
ax.set_ylabel('Salary (GBP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))

ax = axes[1, 0]
sr_df = df[df['salary_range'].notna() & (df['salary_range'] > 0)].copy()
ax.scatter(sr_df['salary_range'].clip(upper=100000), sr_df[SCORE_COL],
           alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('Salary Range (max - min)')
ax.set_ylabel('Risk Score %')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_title('Salary Range vs Fraud Risk Score', fontweight='bold')

ax = axes[1, 1]
miss_sal = df.groupby('source_platform')['salary_missing'].mean() * 100
ax.bar(miss_sal.index, miss_sal.values, color=sns.color_palette('muted', len(miss_sal)))
ax.set_title('Missing Salary Rate by Platform (%)', fontweight='bold')
ax.set_ylabel('% Missing Salary')
for i, (p, val) in enumerate(miss_sal.items()):
    ax.text(i, val+0.5, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '4_salary_analysis.png', bbox_inches='tight')
plt.close()
print("Saved → 4_salary_analysis.png")

In [ ]:
# ANALYSIS 4: TIME-BASED ANALYSIS
print("ANALYSIS 4: TIME-BASED ANALYSIS")

df_time = df[df['posted_date'].notna()].copy()
df_time['posted_month'] = df_time['posted_date'].dt.to_period('M').astype(str)
df_time['posted_dow']   = df_time['posted_date'].dt.day_name()

monthly = df_time.groupby('posted_month').agg(
    total      = (PRED_COL, 'count'),
    fake_count = (PRED_COL, 'sum'),
    avg_risk   = (SCORE_COL, 'mean'),
).reset_index().sort_values('posted_month')
monthly = monthly[monthly['posted_month'] != 'NaT'].tail(12)
monthly['fake_rate_%'] = (monthly['fake_count'] / monthly['total'] * 100).round(1)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df_time.groupby('posted_dow').agg(
    total      = (PRED_COL, 'count'),
    avg_risk   = (SCORE_COL, 'mean'),
    fake_count = (PRED_COL, 'sum'),
).reindex(dow_order).fillna(0)
dow['fake_rate_%'] = (dow['fake_count'] / dow['total'].replace(0,1) * 100).round(1)

df_time['days_since_posted'] = (df_time['collected_date'] - df_time['posted_date']).dt.days

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analysis 4: Time-Based Posting Pattern Analysis',
             fontsize=13, fontweight='bold')

ax = axes[0, 0]; x = range(len(monthly))
ax2 = ax.twinx()
ax.bar(x, monthly['total'], color='steelblue', alpha=0.6, label='Total Jobs')
ax2.plot(x, monthly['avg_risk'], color='tomato', marker='o', lw=2, label='Avg Risk Score')
ax.set_xticks(x)
ax.set_xticklabels(monthly['posted_month'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Number of Jobs', color='steelblue')
ax2.set_ylabel('Avg Risk Score %', color='tomato')
ax.set_title('Monthly Postings & Risk Score Trend', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper left')

ax = axes[0, 1]
ax.plot(x, monthly['fake_rate_%'], color='darkorange', marker='s', lw=2)
ax.fill_between(x, monthly['fake_rate_%'], alpha=0.2, color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(monthly['posted_month'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Fake Job Rate %')
ax.set_title('Fake Job Detection Rate Over Time', fontweight='bold')

ax = axes[1, 0]; ax2 = ax.twinx()
ax.bar(dow_order, dow['total'], color='mediumseagreen', alpha=0.6, label='Total Jobs')
ax2.plot(dow_order, dow['avg_risk'], color='tomato', marker='D', lw=2, label='Avg Risk Score')
ax.set_ylabel('Number of Jobs', color='mediumseagreen')
ax2.set_ylabel('Avg Risk Score %', color='tomato')
ax.set_title('Posting Day of Week vs Risk Score', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)

ax = axes[1, 1]
valid_days = df_time[df_time['days_since_posted'].between(0, 400)]
ax.scatter(valid_days['days_since_posted'], valid_days[SCORE_COL],
           alpha=0.3, s=10, color='steelblue')
z = np.polyfit(valid_days['days_since_posted'].dropna(),
               valid_days[SCORE_COL].dropna(), 1)
p = np.poly1d(z)
x_line = np.linspace(0, 400, 100)
ax.plot(x_line, p(x_line), color='tomato', lw=2, linestyle='--', label='Trend')
ax.set_xlabel('Days Since Posted')
ax.set_ylabel('Risk Score %')
ax.set_title('Job Age vs Fraud Risk Score', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(PLOTS_PATH / '4_time_analysis.png', bbox_inches='tight')
plt.close()
print("Saved → 4_time_analysis.png")

In [ ]:
# TRANSPARENCY FRAMEWORK VISUALISATION
print("\nGenerating transparency framework plot...")

feat_map = {
    'missing_company'      : 'Missing Company',
    'high_salary_flag'     : 'High Salary',
    'salary_missing'       : 'Salary Missing',
    'short_desc'           : 'Short Description',
    'low_lexical_diversity': 'Low Lexical Diversity',
    'suspicious_keywords'  : 'Suspicious Keywords',
    'contact_in_desc'      : 'Informal Contact',
    'vague_location'       : 'Vague Location',
    'contract_missing'     : 'Contract Missing',
    'suspicious_title'     : 'Suspicious Title',
    'title_has_caps'       : 'Title ALL CAPS',
    'wide_salary_range'    : 'Wide Salary Range',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Transparency Framework — AI Fraud Detection\n'
             'Trust & Transparency in AI-Powered Recruitment Platforms',
             fontsize=13, fontweight='bold')

ax = axes[0]
flagged_df = df[df[PRED_COL] == 1]
if len(flagged_df) == 0:
    flagged_df = df[df['risk_tier'].isin(['High','Medium'])]

feat_rates  = {label: flagged_df[col].mean()*100
               for col, label in feat_map.items() if col in flagged_df.columns}
feat_series = pd.Series(feat_rates).sort_values(ascending=True)
colors_f    = ['#F44336' if v > 30 else '#FF9800' if v > 10 else '#4CAF50'
               for v in feat_series.values]
ax.barh(feat_series.index, feat_series.values, color=colors_f)
ax.set_title(f'Feature Presence in Flagged Jobs (%)\n(n={len(flagged_df)} jobs flagged as fake)',
             fontweight='bold')
ax.set_xlabel('% of Flagged Jobs with This Indicator')
ax.axvline(20, color='grey', linestyle='--', alpha=0.5)

ax = axes[1]
conf_counts = df['confidence_label'].value_counts()
colors_conf = {
    'High Confidence Fake': '#F44336',
    'Possibly Suspicious' : '#FF9800',
    'Likely Legitimate'   : '#4CAF50'
}
bars_conf = ax.bar(conf_counts.index, conf_counts.values,
                   color=[colors_conf.get(l,'steelblue') for l in conf_counts.index])
ax.set_title('Job Distribution by Confidence Label\n(DistilBERT Output)', fontweight='bold')
ax.set_ylabel('Number of Jobs')
ax.tick_params(axis='x', rotation=15)
for b, v in zip(bars_conf, conf_counts.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+5,
            str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '4_transparency_framework.png', bbox_inches='tight')
plt.close()
print("Saved → 4_transparency_framework.png")

In [ ]:
# RESEARCH FINDINGS REPORT
print("GENERATING RESEARCH FINDINGS REPORT")

lines = []
lines.append("=" * 65)
lines.append("RESEARCH FINDINGS REPORT")
lines.append("Trust & Transparency in AI-Powered Recruitment Platforms")
lines.append("=" * 65)

lines.append("\n[1] DATASET OVERVIEW")
lines.append(f"  Total jobs analysed       : {len(df):,}")
lines.append(f"  Source platforms          : {', '.join(df['source_platform'].unique())}")
lines.append(f"  Job categories            : {df['category'].nunique()}")
lines.append(f"  Date range (posted)       : {df['posted_month'].min()} to {df['posted_month'].max()}")

lines.append("\n[2] FRAUD DETECTION RESULTS")
lines.append(f"  Model                     : DistilBERT Fine-Tuned + Hybrid Guard")
lines.append(f"  Fake jobs detected        : {df[PRED_COL].sum()} ({df[PRED_COL].mean()*100:.1f}%)")
lines.append(f"  High Confidence Fake      : {df['confidence_label'].eq('High Confidence Fake').sum()}")
lines.append(f"  Possibly Suspicious       : {df['confidence_label'].eq('Possibly Suspicious').sum()}")
lines.append(f"  Likely Legitimate         : {df['confidence_label'].eq('Likely Legitimate').sum()}")
lines.append(f"  High risk tier (rule)     : {(df['risk_tier']=='High').sum()} ({(df['risk_tier']=='High').mean()*100:.1f}%)")
lines.append(f"  Medium risk tier (rule)   : {(df['risk_tier']=='Medium').sum()} ({(df['risk_tier']=='Medium').mean()*100:.1f}%)")

lines.append("\n[3] PLATFORM COMPARISON FINDINGS")
for _, row in plat.iterrows():
    lines.append(f"  {row['source_platform']:12s} — {row['total_jobs']:4d} jobs | "
                 f"fake: {row['fake_rate_%']:.1f}% | avg risk score: {row['avg_risk_score']:.1f}")

lines.append("\n[4] CATEGORY FINDINGS (Top 5 by Risk Score)")
for _, row in cat.nlargest(5, 'avg_risk_score').iterrows():
    lines.append(f"  {row['category'][:35]:35s} — "
                 f"score: {row['avg_risk_score']:.1f} | fake: {row['fake_rate_%']:.1f}%")

lines.append("\n[5] SALARY PATTERN FINDINGS")
lines.append(f"  Overall median salary     : £{df['salary_avg'].median():,.0f}")
lines.append(f"  95th percentile salary    : £{df['salary_avg'].quantile(0.95):,.0f}")
lines.append(f"  Missing salary            : {df['salary_missing'].sum()} ({df['salary_missing'].mean()*100:.1f}%)")
for tier in ['Low','Medium','High']:
    sub = df[df['risk_tier']==tier]['salary_avg'].dropna()
    if len(sub):
        lines.append(f"  {tier} risk median salary  : £{sub.median():,.0f}")

lines.append("\n[6] TIME-BASED FINDINGS")
peak_month = monthly.loc[monthly['total'].idxmax(), 'posted_month']
peak_risk  = monthly.loc[monthly['avg_risk'].idxmax(), 'posted_month']
lines.append(f"  Peak posting month        : {peak_month}")
lines.append(f"  Highest risk month        : {peak_risk}")
lines.append(f"  Most active posting day   : {dow['total'].idxmax()}")

lines.append("\n[7] KEY TRANSPARENCY INDICATORS (in flagged jobs)")
for label, val in feat_series.sort_values(ascending=False).items():
    lines.append(f"  {label:30s}: {val:.1f}%")

lines.append("\n[8] RESEARCH CONTRIBUTION")
lines.append("  DistilBERT fine-tuned on EMSCAD (AUC=0.985) combined with")
lines.append("  a hybrid guard (ML threshold + rule-based score) produces")
lines.append("  realistic fake job detection on UK platform data.")
lines.append("  The transparency framework provides each flagged job with")
lines.append("  named, explainable indicators — directly addressing RQ2 and")
lines.append("  RQ3 on trust and transparency in AI-powered recruitment.")
lines.append("  Findings are consistent with Baraneetharan (2022) and Ch'ng & Yi (2025).")

report = '\n'.join(lines)
print(report)

out = REPORTS_PATH / '4_research_findings.txt'
with open(out, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"\nSaved → {out}")